In [1]:
import os
import hashlib
from collections import Counter
from typing import List, Union
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
import tqdm
from sklearn.metrics import classification_report
from google.colab import drive
import subprocess

# Install timm library if not present
try:
    import timm
except ImportError:
    subprocess.run(["pip", "install", "-q", "timm"])
    import timm

# Install torchmetrics library if not present
try:
    import torchmetrics
except ImportError:
    subprocess.run(["pip", "install", "-q", "torchmetrics"])
    import torchmetrics

# Mount Google Drive
try:
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except ImportError:
    print("Not running in Google Colab. Skipping Drive mount.")

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")


Mounted at /content/drive
Google Drive mounted successfully.
Using device: cuda
GPU Name: Tesla T4


In [2]:

# --- Paths ---
subprocess.run(["unzip", 
    "-q", "/content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip",
    "-d", "/content/Datasets"])
#  unzip -q /content/drive/MyDrive/Datasets/kaggle_knee_osteoarthritis.zip -d /content/Datasets
DATASET_ROOT_PATH = "/content/Datasets/kaggle_knee_osteoarthritis"
CHECKPOINT_SAVE_DIR = "/content/drive/MyDrive/Models/se_resnext50_checkpoints"

os.makedirs(CHECKPOINT_SAVE_DIR, exist_ok=True)

# --- Training Hyperparameters ---
EPOCHS = 100
EPOCHS_STAGE1 = 10  # Max epochs for Stage 1 (Classifier only warm-up)
BATCH_SIZE = 16
IMG_SIZE = 310  # Size specified by paper
INITIAL_LR = 1e-4
WEIGHT_DECAY = 1e-4

# --- Fine-Tuning & Loss Strategy Options ---
ORDINAL_TYPE = "threshold"     # Using Frank-Hall Binary Ordinal Threshold method
FINE_TUNE = True               # True to use Discriminative Fine-Tuning (3 groups) for backbone
EARLY_STOPPING_PATIENCE = 20   # Set to 0 to disable early stopping


In [3]:
class SquarePadOpenCV(object):
    """Pads a rectangular image to a square."""
    def __call__(self, image):
        h, w = image.shape[:2]
        max_wh = max(h, w)
        pad_top = (max_wh - h) // 2
        pad_bottom = max_wh - h - pad_top
        pad_left = (max_wh - w) // 2
        pad_right = max_wh - w - pad_left
        
        padded_image = cv2.copyMakeBorder(
            image, pad_top, pad_bottom, pad_left, pad_right, 
            borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0]
        )
        return padded_image

class OpenCVCLAHE(object):
    """Applies CLAHE (Contrast Limited Adaptive Histogram Equalization) using OpenCV."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clip_limit = clip_limit
        self.tile_grid_size = tile_grid_size

    def __call__(self, img_rgb: np.ndarray) -> np.ndarray:
        clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
        img_lab = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2LAB)
        l_channel, a_channel, b_channel = cv2.split(img_lab)
        clahe_l_channel = clahe.apply(l_channel)
        merged_lab_image = cv2.merge((clahe_l_channel, a_channel, b_channel))
        return cv2.cvtColor(merged_lab_image, cv2.COLOR_LAB2RGB)

def get_transforms(img_size=310):
    """Returns training and validation transforms."""
    train_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.RandomHorizontalFlip(p=0.5), # Standard flip for symmetric joint X-rays
        transforms.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.80, 1.20), shear=5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    
    val_transform = transforms.Compose([
        SquarePadOpenCV(),
        OpenCVCLAHE(),
        transforms.ToPILImage(),
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    return train_transform, val_transform

def remove_duplicate_images(image_paths: List[str], labels: List[int], exclude_hashes: set = None, categories: List[str] = None):
    """Removes duplicate images using MD5 hashing."""
    total_found = len(image_paths)
    unique_paths, unique_labels, unique_hashes = [], [], set()
    internal_dup_count, leakage_count = 0, 0
    
    for path, label in zip(image_paths, labels):
        hash_md5 = hashlib.md5()
        try:
            with open(path, "rb") as f:
                for chunk in iter(lambda: f.read(4096), b""):
                    hash_md5.update(chunk)
            h = hash_md5.hexdigest()
        except Exception as e:
            print(f"Warning: Could not read image {path}: {e}")
            continue
            
        if exclude_hashes and h in exclude_hashes:
            leakage_count += 1
            continue
        if h in unique_hashes:
            internal_dup_count += 1
            continue
            
        unique_hashes.add(h)
        unique_paths.append(path)
        unique_labels.append(label)
        
    class_counts = Counter(unique_labels)
    print(f"\n--- Dataset Statistics & Deduplication ---")
    print(f"  - Total files: {total_found} | Unique kept: {len(unique_paths)}")
    print(f"  - Internal dupes removed: {internal_dup_count} | Cross-split leaks removed: {leakage_count}")
    return unique_paths, unique_labels, unique_hashes

class KaggleKneeOsteoarthritisDataset(Dataset):
    """Dataset class specifically for the Kaggle Knee Osteoarthritis dataset."""
    def __init__(self, root: str, split_dir: str, transform=None, exclude_hashes: set = None):
        self.root = root
        self.transform = transform
        self.exclude_hashes = exclude_hashes
        raw_paths, raw_labels = [], []
        split_path = os.path.join(root, split_dir)
        
        if not os.path.isdir(split_path): 
            raise FileNotFoundError(f"Split directory not found: {split_path}")
            
        class_names = sorted([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d)) and d.isdigit()])
        print(f"Loading '{split_dir}' split from: {split_path}")
        
        for class_name in class_names:
            class_dir = os.path.join(split_path, class_name)
            label = int(class_name)
            valid_extensions = ('.png', '.jpg', '.jpeg')
            image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(valid_extensions)]
            for file_name in image_files:
                raw_paths.append(os.path.join(class_dir, file_name))
                raw_labels.append(label)
                
        self.image_paths, self.labels, self.image_hashes = remove_duplicate_images(
            raw_paths, raw_labels, exclude_hashes=self.exclude_hashes, categories=class_names
        )

    def load_image_from_path(self, image_path: str) -> np.ndarray:
        img_bgr = cv2.imread(image_path)
        if img_bgr is None: raise IOError(f"Could not read image: {image_path}")
        return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    def __getitem__(self, idx: int):
        image = self.load_image_from_path(self.image_paths[idx])
        label = self.labels[idx]
        if self.transform: image = self.transform(image)
        return image, label

    def __len__(self) -> int: 
        return len(self.image_paths)


In [4]:
# --- 1. Prepare Data Loaders (Class-Aware Sampling) ---
train_transform_standard, train_transform_strong, val_transform = get_transforms(img_size=IMG_SIZE)

train_dataset = KaggleKneeOsteoarthritisDataset(
    root=DATASET_ROOT_PATH, split_dir="train", 
    transform=train_transform_standard, strong_transform=train_transform_strong
)
train_hashes = set(train_dataset.image_hashes)

val_split_dir = "val" if os.path.isdir(os.path.join(DATASET_ROOT_PATH, "val")) else "test"
val_dataset = KaggleKneeOsteoarthritisDataset(
    root=DATASET_ROOT_PATH, split_dir=val_split_dir, 
    transform=val_transform, exclude_hashes=train_hashes
)

# Dynamic split if val dataset is empty after leakage removal
if len(val_dataset) == 0:
    import random
    print("Performing dynamic 80/20 train/validation split...")
    combined = list(zip(train_dataset.image_paths, train_dataset.labels))
    random.seed(42)
    random.shuffle(combined)
    split_idx = int(len(combined) * 0.8)
    train_pairs, val_pairs = combined[:split_idx], combined[split_idx:]
    
    train_dataset.image_paths, train_dataset.labels = [p for p, _ in train_pairs], [l for _, l in train_pairs]
    val_dataset.image_paths, val_dataset.labels = [p for p, _ in val_pairs], [l for _, l in val_pairs]
    print(f"Post-Split - Train: {len(train_dataset)}, Val: {len(val_dataset)}")

# Configure WeightedRandomSampler to balance classes dynamically in each batch
from torch.utils.data import WeightedRandomSampler
counts = Counter(train_dataset.labels)
print(f"Train class distributions for sampler calculation: {dict(sorted(counts.items()))}")

class_weights = {c: 1.0 / count for c, count in counts.items()}
sample_weights = [class_weights[label] for label in train_dataset.labels]
sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# Note: When sampler is used, shuffle must be False (sampler handles randomness)
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)


Loading 'train' split from: /content/Datasets/kaggle_knee_osteoarthritis/train

--- Dataset Statistics & Deduplication ---
  - Total files: 5778 | Unique kept: 5778
  - Internal dupes removed: 0 | Cross-split leaks removed: 0
Loading 'val' split from: /content/Datasets/kaggle_knee_osteoarthritis/val

--- Dataset Statistics & Deduplication ---
  - Total files: 826 | Unique kept: 826
  - Internal dupes removed: 0 | Cross-split leaks removed: 0


In [5]:
class SEResNeXtModel(nn.Module):
    def __init__(self, num_classes: int = 5, pretrained: bool = True, ordinal_type: str = "threshold"):
        super(SEResNeXtModel, self).__init__()
        self.ordinal_type = ordinal_type
        self.num_classes = num_classes
        # In Threshold Ordinal classification, the output layer has (num_classes - 1) nodes.
        out_features = num_classes - 1 if ordinal_type == "threshold" else num_classes
        self.model = timm.create_model('seresnext50_32x4d', pretrained=pretrained, num_classes=out_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x)

    def freeze_backbone(self):
        """Freezes backbone layers, leaving only the classifier fc layer trainable."""
        print("Applying standard freezing strategy for SE-ResNeXt-50.")
        for param in self.model.parameters():
            param.requires_grad = False
        for param in self.model.fc.parameters():
            param.requires_grad = True

    def fit(self, epoch, data_loader, optimizer, device, pos_weights, scheduler=None):
        self.to(device)
        self.train()
        running_loss, total, correct = 0.0, 0, 0
        
        progress_bar = tqdm.tqdm(data_loader, desc=f"Epoch {epoch+1} [TRAIN]")
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = self(images)

            # Calculate Frank-Hall Threshold Loss (Average BCE across 4 sub-tasks)
            loss = 0.0
            num_tasks = self.num_classes - 1
            alpha = 0.05 # Label smoothing factor for binary tasks
            for j in range(num_tasks):
                targets_j = (labels > j).float()
                # Apply binary label smoothing: 1 -> 0.975, 0 -> 0.025
                targets_j_smoothed = targets_j * (1.0 - alpha) + 0.5 * alpha
                loss += F.binary_cross_entropy_with_logits(output[:, j], targets_j_smoothed, pos_weight=pos_weights[j])
            loss = loss / num_tasks

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            
            # Ordinal class prediction: sum of binary tasks exceeded threshold 0.5
            probs = torch.sigmoid(output)
            predicted = (probs > 0.5).sum(dim=1)
            
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # Track all learning rates for custom display
            lrs = [f"{pg['lr']:.1e}" for pg in optimizer.param_groups]
            lr_str = ", ".join(lrs)
            progress_bar.set_postfix({
                "loss": f"{loss.item():.4f}",
                "acc": f"{100.0 * correct / total:.2f}%",
                "lr": lr_str
            })
            
        return running_loss / total, 100.0 * correct / total

    def evaluate(self, epoch, data_loader, device):
        self.to(device)
        self.eval()
        running_loss, total, correct = 0.0, 0, 0
        all_preds, all_labels = [], []
        
        progress_bar = tqdm.tqdm(data_loader, desc=f"Epoch {epoch+1} [VALIDATE]")
        with torch.no_grad():
            for images, labels in progress_bar:
                images, labels = images.to(device), labels.to(device)
                output = self(images)

                # Compute validation loss (Threshold BCE)
                loss = 0.0
                num_tasks = self.num_classes - 1
                for j in range(num_tasks):
                    targets_j = (labels > j).float()
                    loss += F.binary_cross_entropy_with_logits(output[:, j], targets_j)
                loss = loss / num_tasks

                running_loss += loss.item() * images.size(0)
                
                # Predict class
                probs = torch.sigmoid(output)
                predicted = (probs > 0.5).sum(dim=1)
                
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        report = classification_report(
            all_labels, all_preds, 
            target_names=[str(i) for i in range(5)], 
            zero_division=0
        )
        
        # Calculate and print Cohen's Quadratic Weighted Kappa
        from torchmetrics.classification import CohenKappa
        kappa_metric = CohenKappa(task="multiclass", num_classes=5, weights="quadratic")
        kappa_score = kappa_metric(torch.tensor(all_preds), torch.tensor(all_labels)).item()
        print(f"\nVal Quadratic Weighted Kappa: {kappa_score:.4f}")
        
        return running_loss / total, 100.0 * correct / total, report


In [6]:
# --- 2. Initialize Model ---
model = SEResNeXtModel(num_classes=5, pretrained=True, ordinal_type=ORDINAL_TYPE)

# Calculate binary threshold positive class weights dynamically with square-root dampening
from collections import Counter
counts = Counter(train_dataset.labels)
print(f"Train class distributions: {dict(sorted(counts.items()))}")

num_classes_minus_1 = 4
pos_weights_list = []
for j in range(num_classes_minus_1):
    neg = sum(counts[i] for i in range(j + 1))
    pos = sum(counts[i] for i in range(j + 1, 5))
    # Take the square root of the negative-to-positive ratio to prevent gradient dominance
    pos_weight = (neg / pos if pos > 0 else 1.0) ** 0.5
    pos_weights_list.append(pos_weight)

pos_weights = torch.tensor(pos_weights_list, dtype=torch.float32, device=device)
model.pos_weights = pos_weights
print(f"Calculated binary threshold pos_weights (dampened via square root): {pos_weights_list}")

# --- 3. Helper Functions for Stage setups ---
def setup_stage1(model):
    print("=== [STAGE 1] Freeze Backbone, Train Classifier Head Only ===")
    model.freeze_backbone()
    # Optimizer only updates classifier
    optimizer = optim.AdamW(model.model.fc.parameters(), lr=INITIAL_LR, weight_decay=WEIGHT_DECAY)
    return optimizer

def setup_stage2(model):
    print("=== [STAGE 2] Unfreeze Backbone & Fine-Tune Model ===")
    if not FINE_TUNE:
        for param in model.parameters():
            param.requires_grad = True
        optimizer = optim.AdamW(model.parameters(), lr=INITIAL_LR, weight_decay=WEIGHT_DECAY)
    else:
        print("Applying Discriminative Fine-Tuning (3 groups) for SE-ResNeXt-50")
        early_params, late_params, classifier_params = [], [], []
        for n, p in model.named_parameters():
            if 'fc' in n:
                classifier_params.append(p)
            elif any(layer_name in n for layer_name in ['layer3', 'layer4']):
                late_params.append(p)
            else:
                early_params.append(p)
                
        optimizer = optim.AdamW([
            {'params': early_params, 'lr': INITIAL_LR * 0.01},
            {'params': late_params, 'lr': INITIAL_LR * 0.1},
            {'params': classifier_params, 'lr': INITIAL_LR}
        ], weight_decay=WEIGHT_DECAY)
        print(f"Discriminative LRs -> Early: {INITIAL_LR * 0.01}, Late: {INITIAL_LR * 0.1}, Head: {INITIAL_LR}")
    return optimizer

# --- 4. Define Checkpoint Paths ---
last_model_path = os.path.join(CHECKPOINT_SAVE_DIR, "last_model.pth")
best_model_stage1_path = os.path.join(CHECKPOINT_SAVE_DIR, "best_model_stage1.pth")
best_model_stage2_path = os.path.join(CHECKPOINT_SAVE_DIR, "best_model.pth")

# --- 5. Resume from Checkpoint (if exists) ---
current_stage = 1
current_epoch = 0
val_loss_min_stage1 = np.inf
val_loss_min_stage2 = np.inf
early_stop_counter_stage2 = 0

if os.path.exists(last_model_path):
    print(f"Loading local checkpoint from: {last_model_path}")
    try:
        checkpoint = torch.load(last_model_path, map_location=device)
        model.load_state_dict(checkpoint["model"])
        current_stage = checkpoint.get("stage", 1)
        current_epoch = checkpoint.get("epoch", 0) + 1
        
        if current_stage == 1:
            val_loss_min_stage1 = checkpoint.get("val_loss_min", np.inf)
        else:
            val_loss_min_stage2 = checkpoint.get("val_loss_min", np.inf)
            early_stop_counter_stage2 = checkpoint.get("early_stop_counter", 0)
            
        if "rng_state" in checkpoint: torch.set_rng_state(checkpoint["rng_state"].cpu())
        if "cuda_rng_state" in checkpoint and torch.cuda.is_available():
            try: torch.cuda.set_rng_state_all([s.cpu() for s in checkpoint["cuda_rng_state"]])
            except Exception: pass
        print(f"Successfully resumed from Stage {current_stage}, Epoch {current_epoch}.")
    except Exception as e:
        print(f"Could not load checkpoint ({e}). Starting from scratch.")


WeightedRandomSampler is active. pos_weights set to None.
Loading local checkpoint from: /content/drive/MyDrive/Models/se_resnext50_checkpoints/last_model.pth
Successfully resumed from Stage 2, Epoch 100.


In [7]:

class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=7, verbose=False, delta=0, path='checkpoint.pt'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf
        self.delta = delta
        self.path = path

    def __call__(self, val_loss, model, optimizer, scheduler, epoch):
        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, scheduler, epoch)
        elif score < self.best_score + self.delta:
            self.counter += 1
            print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, scheduler, epoch)
            self.counter = 0
        return self.early_stop

    def save_checkpoint(self, val_loss, model, optimizer, scheduler, epoch):
        if self.verbose:
            print(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}). Saving model...')
        checkpoint = {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler else None,
            "epoch": epoch,
            "val_loss": val_loss
        }
        torch.save(checkpoint, self.path)
        self.val_loss_min = val_loss

# --- STAGE 1: Train Classifier Only (Warm-up) ---
if current_stage == 1:
    optimizer = setup_stage1(model)
    # Cosine Annealing scheduler updates after every epoch
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=1e-6)
    
    # Load optimizer state if resuming in Stage 1
    if os.path.exists(last_model_path):
        try:
            checkpoint = torch.load(last_model_path, map_location=device)
            if "optimizer" in checkpoint: optimizer.load_state_dict(checkpoint["optimizer"])
            if "scheduler" in checkpoint: 
                try: scheduler.load_state_dict(checkpoint["scheduler"])
                except Exception: pass
            for state in optimizer.state.values():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor): state[k] = v.to(device)
        except Exception as e:
            print(f"Could not load Stage 1 optimizer: {e}")
            
    for epoch in range(current_epoch, EPOCHS_STAGE1):
        print(f"\n--- [STAGE 1] Epoch {epoch+1}/{EPOCHS_STAGE1} ---")
        train_loss, train_acc = model.fit(epoch, train_loader, optimizer, device, pos_weights, scheduler)
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")

        val_loss, val_acc, report = model.evaluate(epoch, val_loader, device)
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(report)
        
        scheduler.step()
        
        # Save last model (Atomic) for Stage 1
        checkpoint = {
            "model": model.state_dict(), 
            "optimizer": optimizer.state_dict(), 
            "scheduler": scheduler.state_dict(), 
            "epoch": epoch, 
            "stage": 1,
            "val_loss": val_loss,
            "val_loss_min": val_loss_min_stage1,
            "rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        }
        tmp_path = f"{last_model_path}.tmp"
        torch.save(checkpoint, tmp_path)
        if os.path.exists(tmp_path): os.replace(tmp_path, last_model_path)
        
        # Save best model weights when validation loss decreases
        if val_loss < val_loss_min_stage1:
            print(f"Validation loss decreased ({val_loss_min_stage1:.6f} --> {val_loss:.6f}). Saving best Stage 1 model...")
            val_loss_min_stage1 = val_loss
            best_checkpoint = {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "epoch": epoch,
                "val_loss": val_loss
            }
            torch.save(best_checkpoint, best_model_stage1_path)
    print("\nStage 1 finished. Loading best Stage 1 checkpoint and moving to Stage 2 fine-tuning...")
    if os.path.exists(best_model_stage1_path):
        try:
            checkpoint = torch.load(best_model_stage1_path, map_location=device)
            model.load_state_dict(checkpoint["model"])
            print("Successfully loaded best Stage 1 model weights.")
        except Exception as e:
            print(f"Could not load best Stage 1 checkpoint: {e}")
            
    # Transition to Stage 2
    current_stage = 2
    current_epoch = 0
    if os.path.exists(last_model_path):
        try: os.remove(last_model_path)
        except Exception: pass


In [8]:

# --- STAGE 2: Fine-Tuning ---
if current_stage == 2:
    optimizer = setup_stage2(model)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=1, eta_min=1e-6)
    
    # Load optimizer state if resuming in Stage 2
    if os.path.exists(last_model_path):
        try:
            checkpoint = torch.load(last_model_path, map_location=device)
            if "optimizer" in checkpoint: optimizer.load_state_dict(checkpoint["optimizer"])
            if "scheduler" in checkpoint: 
                try: scheduler.load_state_dict(checkpoint["scheduler"])
                except Exception: pass
            for state in optimizer.state.values():
                for k, v in state.items():
                    if isinstance(v, torch.Tensor): state[k] = v.to(device)
        except Exception as e:
            print(f"Could not load Stage 2 optimizer: {e}")
            
    early_stopper = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, verbose=True, path=best_model_stage2_path)
    early_stopper.val_loss_min = val_loss_min_stage2
    early_stopper.best_score = -val_loss_min_stage2
    early_stopper.counter = early_stop_counter_stage2
    
    for epoch in range(current_epoch, EPOCHS):
        print(f"\n--- [STAGE 2] Epoch {epoch+1}/{EPOCHS} ---")
        train_loss, train_acc = model.fit(epoch, train_loader, optimizer, device, pos_weights, scheduler)
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")

        val_loss, val_acc, report = model.evaluate(epoch, val_loader, device)
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(report)
        
        scheduler.step()
        
        # Save last model (Atomic) for Stage 2
        checkpoint = {
            "model": model.state_dict(), 
            "optimizer": optimizer.state_dict(), 
            "scheduler": scheduler.state_dict(), 
            "epoch": epoch, 
            "stage": 2,
            "val_loss": val_loss,
            "val_loss_min": early_stopper.val_loss_min,
            "early_stop_counter": early_stopper.counter,
            "rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None
        }
        tmp_path = f"{last_model_path}.tmp"
        torch.save(checkpoint, tmp_path)
        if os.path.exists(tmp_path): os.replace(tmp_path, last_model_path)
        
        if early_stopper(val_loss, model, optimizer, scheduler, epoch):
            print("Stage 2 Early stopping triggered!")
            break

# Disconnect Colab runtime to save credits after training finishes
try:
    from google.colab import runtime
    print("Training complete. Disconnecting runtime...")
    runtime.unassign()
except ImportError:
    print("Not running in Colab. Skip unassign.")


=== [STAGE 2] Unfreeze Backbone & Fine-Tune Model ===
Applying Discriminative Fine-Tuning (3 groups) for SE-ResNeXt-50
Discriminative LRs -> Early: 1.0000000000000002e-06, Late: 1e-05, Head: 0.0001
Training complete. Disconnecting runtime...
